In [1]:
from looptrace import ImageHandler
import dask.array as da
import numpy as np
import pandas as pd
from looptrace import image_processing_functions as ip
from skimage.registration import phase_cross_correlation
import napari
import os
from skimage.measure import regionprops_table
from numba import njit
%load_ext autoreload
%autoreload 2

In [29]:
def coord_to_shape(coord, shape):
    diff = (shape - (coord[1]-coord[0]))//2
    return (coord[0]-diff, coord[1]+diff)

In [3]:
H = ImageHandler(r"P:\Kai\2021-10-29_KSB020_NikTrace_4x10pos_noRNAse_A643imagers\analysis\2021-10-29_KSB020_1D_4x10_config.yaml")
P, T, C, Z, Y, X = H.images.shape

Images loaded from multiple tiff folders:  dask.array<concatenate, shape=(20, 21, 2, 57, 2304, 2304), dtype=uint16, chunksize=(1, 1, 2, 1, 2304, 2304), chunktype=numpy.ndarray>


In [37]:
ref_frame = H.config['bead_reference_frame']
ref_ch = H.config['bead_ch']
ds = H.config['course_drift_downsample']
z = 20
z_mito = 40
drift_res =[]
for p in range(1):
    t_img = H.images[p, ref_frame, ref_ch, z, ::ds, ::ds]
    for t in range(T):
        o_img = H.images[p, t, ref_ch, z, ::ds, ::ds]
        course_drift = phase_cross_correlation(np.array(t_img), np.array(o_img), return_error=False) * ds
        drift_res.append([p, t, *course_drift])
drift_res = pd.DataFrame(drift_res, columns=['position', 'frame', 'y_course', 'x_course'])
drift_res

,position,frame,y_course,x_course
0,0,0,-4.0,0.0
1,0,1,0.0,2.0
2,0,2,0.0,0.0
3,0,3,0.0,0.0
4,0,4,2.0,-2.0
5,0,5,2.0,-2.0
6,0,6,4.0,-4.0
7,0,7,4.0,-4.0
8,0,8,8.0,-6.0
9,0,9,6.0,-4.0


In [24]:
nuc_ch = H.config['nuc_channel']
nuc_frame = H.config['nuc_ref_frame']
nuc_images = [np.array(H.images[p,nuc_frame,nuc_ch,z,:,:]) for p in range(P)]
mito_images = [np.array(H.images[p,nuc_frame,nuc_ch,z_mito,:,:]) for p in range(P)]

In [25]:
nuc_masks = ip.nuc_segmentation(nuc_images, model='nuclei')

2021-11-02 19:12:40,731 [INFO] >>>> using CPU
2021-11-02 19:12:40,785 [INFO] ~~~ FINDING MASKS ~~~
2021-11-02 19:13:22,485 [INFO] 100%|##########| 20/20 [00:41<00:00,  2.08s/it]
2021-11-02 19:13:22,489 [INFO] >>>> TOTAL TIME 41.70 sec


In [26]:
nuc_masks, mitotic_idx = zip(*[ip.mitotic_cell_extra_seg(mito_images[i], nuc_masks[i]) for i in range(len(nuc_images))])
nuc_class = []
for i, nuc_mask in enumerate(nuc_masks):
    class_1 = ((nuc_mask > 0) & (nuc_mask < mitotic_idx[i])).astype(int)
    class_2 = (nuc_mask >= mitotic_idx[i]).astype(int)
    nuc_class.append(class_1 + class_2*2)

nuc_images = np.stack(nuc_images)
nuc_masks = np.stack(nuc_masks)
nuc_class = np.stack(nuc_class)
#nucs = np.stack([np.stack(nuc_images), np.stack(nuc_masks)])
#np.save(H.config['output_path']+os.sep+H.config['output_prefix']+'nucs.npy', nucs)

In [27]:
n = napari.view_image(nuc_images)
nuc_masks = n.add_labels(nuc_masks).data
nuc_class = n.add_labels(nuc_class).data

2021-11-02 19:13:33,019 [WARNING] QWindowsWindow::setGeometry: Unable to set geometry 1280x1214+0+34 (frame: 1302x1270-11-11) on QWidgetWindow/"_QtMainWindowClassWindow" on "\\.\DISPLAY257". Resulting geometry: 1280x1092+0+34 (frame: 1302x1148-11-11) margins: 11, 45, 11, 11 minimum size: 608x546 MINMAXINFO maxSize=0,0 maxpos=0,0 mintrack=1238,1148 maxtrack=0,0)


In [32]:
if not np.allclose(n_masks.data, nuc_masks):
    print('Labels changed.')
    nucs[1] = n_masks.data
    np.save(H.config['output_path']+os.sep+H.config['output_prefix']+'nucs.npy', nucs)

In [28]:
nuc_props = pd.DataFrame()
for p in range(P):
    nuc_props_p = pd.DataFrame(regionprops_table(label_image = nuc_masks[p], intensity_image = nuc_class[p], properties=('label','bbox')))
    nuc_props_p['fov'] = p
    nuc_props = nuc_props.append(nuc_props_p)
nuc_props = nuc_props.reset_index(drop=True)
print(nuc_props)

      label  bbox-0  bbox-1  bbox-2  bbox-3  fov
0         1       0    1431      89    1581    0
1         2       0    1183     133    1289    0
2         3      18    2004     177    2181    0
3         4      53     698     186     883    0
4         5     151    1563     283    1713    0
...     ...     ...     ...     ...     ...  ...
1218     60    2225     583    2304     716   19
1219     61    2243       0    2304      89   19
1220     62     134    2068     209    2144   19
1221     63     660    1966     757    2065   19
1222     64    1660     592    1770     709   19

[1223 rows x 6 columns]


In [31]:
import numpy as np
import tqdm
import dask.array as da

box_size = 200

imgs = []

for p in range(P):
    for t in range(T):
        for i, nuc in tqdm.tqdm(nuc_props.query('fov == @p').iterrows()):
            #p = nuc['fov']
            y = coord_to_shape((nuc['bbox-0'], nuc['bbox-2']), box_size)
            x = coord_to_shape((nuc['bbox-1'], nuc['bbox-3']), box_size)
            dy, dx = drift_res.query('position == @p & frame == @t')[['y_course', 'x_course']].to_list()
            ymin = max(0, y[0])
            ymax = min(Y, y[1])
            xmin = max(0, x[0])
            xmax = min(X, x[1])
            seg_image = H.images[p,:,:,:,ymin:ymax,xmin:xmax]
            s = seg_image.shape
            seg_image = da.pad(seg_image, ((0,0), (0,0), (0,0), (box_size-s[-2], 0), (box_size-s[-1], 0)))
            seg_image = da.rechunk(seg_image, chunks=(1,1,Z,box_size,box_size))
            #print(nuc_image.shape)
            imgs.append(seg_image)
imgs = da.stack(imgs)


1223it [01:05, 18.60it/s]


In [33]:
from numcodecs import Blosc
compressor = Blosc(cname='zstd', clevel=5, shuffle=Blosc.BITSHUFFLE)
da.to_zarr(imgs, r'P:\Kai\2021-10-29_KSB020_NikTrace_4x10pos_noRNAse_A643imagers\nuc_zarr', compressor=compressor)

KeyboardInterrupt: 

In [35]:
napari.view_image(imgs[5])

2021-11-02 19:39:11,189 [WARNING] QWindowsWindow::setGeometry: Unable to set geometry 1280x1214+0+34 (frame: 1302x1270-11-11) on QWidgetWindow/"_QtMainWindowClassWindow" on "\\.\DISPLAY257". Resulting geometry: 1280x1092+0+34 (frame: 1302x1148-11-11) margins: 11, 45, 11, 11 minimum size: 608x546 MINMAXINFO maxSize=0,0 maxpos=0,0 mintrack=1238,1148 maxtrack=0,0)


Viewer(axes=Axes(visible=False, labels=True, colored=True, dashed=False, arrows=True), camera=Camera(center=(0.0, 99.5, 99.5), zoom=1.670854271356784, angles=(0.0, 0.0, 90.0), perspective=0.0, interactive=True), cursor=Cursor(position=(1.0, 1.0, 0.0, 0.0, 0.0), scaled=True, size=1, style=<CursorStyle.STANDARD: 'standard'>), dims=Dims(ndim=5, ndisplay=2, last_used=4, range=((0.0, 20.0, 1.0), (0.0, 1.0, 1.0), (0.0, 56.0, 1.0), (0.0, 199.0, 1.0), (0.0, 199.0, 1.0)), current_step=(0, 0, 0, 0, 0), order=(0, 1, 2, 3, 4), axis_labels=('0', '1', '2', '3', '4')), grid=GridCanvas(stride=1, shape=(-1, -1), enabled=False), layers=[<Image layer 'Image' at 0x2ebe7a0de80>], scale_bar=ScaleBar(visible=False, colored=False, ticks=True, position=<Position.BOTTOM_RIGHT: 'bottom_right'>, font_size=10.0, unit=None), text_overlay=TextOverlay(visible=False, color=array([0.5, 0.5, 0.5, 1. ]), font_size=10.0, position=<TextOverlayPosition.TOP_LEFT: 'top_left'>, text=''), help='', status='Ready', tooltip=Toolti